# 08 - Aspect-Based Sentiment Analysis

This notebook analyzes sentiment discrepancies between public (Reddit + WebMD) and media (News) corpora across 7 defined aspects of GLP-1 discourse:
- Side Effects
- Efficacy
- Cost
- Access
- Mental Health
- Food Relationship
- Dosage

We tag documents via keyword matching, compute per-aspect sentiment by source, test for statistical significance, and build a logistic regression classifier to predict source from aspect-sentiment features.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
import json

DATA_DIR = '../data'
FIG_DIR = '../figures'

df_public = pd.read_csv(f'{DATA_DIR}/public_with_sentiment.csv')
df_media = pd.read_csv(f'{DATA_DIR}/media_with_sentiment.csv')

print(f'Public corpus: {len(df_public)} docs')
print(f'Media corpus: {len(df_media)} docs')

In [ ]:
aspect_keywords = {
    'side_effects': ['nausea', 'vomiting', 'diarrhea', 'constipation', 'headache', 'fatigue',
                     'hair loss', 'pancreatitis', 'gastroparesis', 'sulfur burp', 'dizz', 'bloat'],
    'efficacy': ['weight loss', 'pound', 'a1c', 'effective', 'result', 'progress', 'lost weight',
                 'blood sugar', 'bmi'],
    'cost': ['cost', 'price', 'expensive', 'insurance', 'coverage', 'afford', 'coupon', 'copay',
             'out of pocket'],
    'access': ['shortage', 'supply', 'pharmacy', 'prescription', 'prior authorization',
               'backorder', 'available', 'stock'],
    'mental_health': ['depression', 'anxiety', 'mood', 'suicidal', 'brain fog', 'mental health',
                      'emotional'],
    'food_relationship': ['appetite', 'food noise', 'craving', 'hunger', 'binge', 'eating',
                          'food aversion'],
    'dosage': ['dose', 'titration', 'injection', 'pen', 'needle', 'milligram', 'weekly shot']
}


def tag_aspects(text):
    text_lower = str(text).lower()
    matched = []
    for aspect, keywords in aspect_keywords.items():
        if any(kw in text_lower for kw in keywords):
            matched.append(aspect)
    return matched


df_all = pd.concat([
    df_public[['text', 'sentiment', 'source']],
    df_media[['text', 'sentiment', 'source']]
], ignore_index=True)
df_all['aspects'] = df_all['text'].apply(tag_aspects)
df_all['corpus'] = df_all['source'].map({'reddit': 'public', 'webmd': 'public', 'news': 'media'})

# How many docs match each aspect
for aspect in aspect_keywords:
    n = df_all['aspects'].apply(lambda x: aspect in x).sum()
    print(f'{aspect}: {n} documents')

In [ ]:
# Explode aspects and compute per-aspect per-corpus sentiment
rows = []
for _, row in df_all.iterrows():
    for aspect in row['aspects']:
        rows.append({'aspect': aspect, 'sentiment': row['sentiment'],
                     'source': row['source'], 'corpus': row['corpus']})
df_asp = pd.DataFrame(rows)

aspect_means = df_asp.groupby(['aspect', 'corpus'])['sentiment'].mean().unstack(fill_value=0)
aspect_means['discrepancy'] = aspect_means['public'] - aspect_means['media']
print(aspect_means.round(4))

In [ ]:
# Per-aspect Mann-Whitney U tests
for aspect in aspect_means.index:
    pub = df_asp[(df_asp['aspect'] == aspect) & (df_asp['corpus'] == 'public')]['sentiment']
    med = df_asp[(df_asp['aspect'] == aspect) & (df_asp['corpus'] == 'media')]['sentiment']
    if len(pub) >= 5 and len(med) >= 5:
        u, p = stats.mannwhitneyu(pub, med, alternative='two-sided')
        sig = '*' if p < 0.05 else ''
        print(f'{aspect:20s}: U={u:8.1f}, p={p:.4f} {sig}  (pub_n={len(pub)}, med_n={len(med)})')
    else:
        print(f'{aspect:20s}: insufficient data (pub={len(pub)}, med={len(med)})')

In [ ]:
# Figure: Grouped bar chart
aspects_sorted = aspect_means.sort_values('discrepancy').index.tolist()
x = np.arange(len(aspects_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, [aspect_means.loc[a, 'public'] for a in aspects_sorted],
       width, label='Public', color='#2196F3', alpha=0.8)
ax.bar(x + width/2, [aspect_means.loc[a, 'media'] for a in aspects_sorted],
       width, label='Media', color='#FF9800', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([a.replace('_', ' ').title() for a in aspects_sorted], rotation=30, ha='right')
ax.set_ylabel('Mean VADER Sentiment')
ax.set_title('Aspect-Based Sentiment: Public vs Media', fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/aspect_sentiment_comparison.png', dpi=150)
plt.show()

In [ ]:
# Figure: Diverging bar chart - discrepancy with significance stars
fig, ax = plt.subplots(figsize=(10, 6))
disc_vals = [aspect_means.loc[a, 'discrepancy'] for a in aspects_sorted]
colors = ['#2196F3' if v >= 0 else '#F44336' for v in disc_vals]
ax.barh(range(len(aspects_sorted)), disc_vals, color=colors, alpha=0.8)
ax.set_yticks(range(len(aspects_sorted)))
ax.set_yticklabels([a.replace('_', ' ').title() for a in aspects_sorted])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sentiment Discrepancy (Public - Media)')
ax.set_title('Public vs Media Sentiment Gap by Aspect', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/aspect_discrepancy.png', dpi=150)
plt.show()

In [ ]:
# Discrepancy classifier: Logistic Regression
aspect_list = sorted(aspect_keywords.keys())
feature_rows = []
labels = []
for _, row in df_all.iterrows():
    feat = {f'has_{a}': int(a in row['aspects']) for a in aspect_list}
    feat['sentiment'] = row['sentiment']
    feature_rows.append(feat)
    labels.append(row['corpus'])

df_feat = pd.DataFrame(feature_rows)
X = df_feat.values
y = np.array(labels)

lr = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='accuracy')
print(f'Logistic Regression 5-fold CV accuracy: {scores.mean():.4f}')
print(f'Per-fold: {[round(s, 4) for s in scores]}')

lr.fit(X, y)
coef_names = [f'has_{a}' for a in aspect_list] + ['sentiment']
for name, coef in zip(coef_names, lr.coef_[0]):
    print(f'  {name:25s}: {coef:+.4f}')